# Results

Loads `results/results.csv` (written by `algo results`) and redoes the tables and charts from the README. Run `uv run algo fetch` and `uv run algo results` first.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../results/results.csv")
print(len(df) // 3, "orders")
df.groupby("algo")[["vwap_slippage_bps", "arrival_slippage_bps", "shortfall_bps", "tracking_error", "fill_rate"]].mean()

## Slippage vs interval VWAP, per order

Each dot is one parent order. Closer to 0 means the algo tracked the market's VWAP more closely.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for i, (algo, g) in enumerate(df.groupby("algo")):
    ax.scatter([i] * len(g), g["vwap_slippage_bps"], alpha=0.5, label=algo)
ax.axhline(0, color="grey", lw=0.8)
ax.set_xticks(range(df.algo.nunique()), sorted(df.algo.unique()))
ax.set_ylabel("slippage vs interval VWAP (bps)")
plt.show()

## VWAP vs TWAP on the same order

In [ ]:
piv = df.pivot_table(index=["symbol", "day"], columns="algo", values="vwap_slippage_bps")
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(piv["twap"].abs(), piv["vwap"].abs())
lim = max(piv["twap"].abs().max(), piv["vwap"].abs().max()) * 1.05
ax.plot([0, lim], [0, lim], color="grey", lw=0.8)
ax.set_xlabel("|TWAP slippage| (bps)")
ax.set_ylabel("|VWAP slippage| (bps)")
print(f"VWAP closer on {(piv['vwap'].abs() < piv['twap'].abs()).sum()} of {len(piv)} orders")
plt.show()

## Where does TWAP lose? Tracking error vs slippage

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for algo, g in df.groupby("algo"):
    ax.scatter(g["tracking_error"] * 100, g["vwap_slippage_bps"].abs(), label=algo, alpha=0.6)
ax.set_xlabel("tracking error vs market volume curve (%)")
ax.set_ylabel("|slippage vs VWAP| (bps)")
ax.legend()
plt.show()